# 03 - Gold: the star schema

Reads the `slv_*` tables and writes a Power BI ready star schema.

| Dimensions | Facts |
|---|---|
| `gld_dim_date` | `gld_fact_sales` (transaction grain) |
| `gld_dim_product` | `gld_fact_plan` (month grain, Budget + Forecast stacked) |
| `gld_dim_market` | `gld_fact_inventory` (month grain) |
| `gld_dim_customer` | `gld_fact_opex` (month grain) |
| `gld_dim_scenario` | |
| `gld_dim_sales_channel` | |
| `gld_dim_warehouse` | |
| `gld_dim_expense_category` | |

## The load policy, and why it is not just "drop the quarantine rows"

Silver classified rows; Gold has to decide what to do with them. Blanket-filtering
`DataQualityStatus <> 'Quarantine'` is the obvious move and it is **half right**:

- **Duplicates must go.** Keeping them double-counts revenue. Removed here, permanently.
- **Unattributable rows must stay.** The 45 sales rows with no ProductID or MarketID carry
  EUR 1.57m of real revenue. Dropping them makes the P&L quietly understate by 0.77% with nothing
  on screen to say so. They are routed to each dimension's **Unknown member** (`-1`) instead, so
  the total still ties and the gap is visible as a slice called `Unknown` that someone can chase.

That is the standard warehouse answer to a missing key, and it is the difference between a report
that is tidy and one that is true. `GOLD_LOAD_POLICY` below lets you switch to the strict version
and watch the total move.

Returns and volume outliers pass straight through - they were never defects.

## 1. Parameters

In [ ]:
# PARAMETERS -----------------------------------------------------------------
SILVER_PREFIX = "slv_"
GOLD_PREFIX   = "gld_"

# "route_unknown"      - drop duplicates, send unresolved keys to the Unknown member (recommended)
# "exclude_quarantine" - drop every row Silver quarantined (totals will understate)
GOLD_LOAD_POLICY = "route_unknown"

DATE_FROM = "2025-01-01"
DATE_TO   = "2026-12-31"

UNKNOWN_KEY = -1
# -----------------------------------------------------------------------------

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import StructType, StructField, StringType

for key, value in {
    "spark.sql.parquet.vorder.enabled": "true",
    "spark.microsoft.delta.optimizeWrite.enabled": "true",
}.items():
    try:
        spark.conf.set(key, value)
    except Exception as exc:
        print(f"[info] could not set {key}: {exc}")

GOLD_TABLES = []
print(f"Load policy: {GOLD_LOAD_POLICY}")

## 2. Helpers

**Surrogate keys.** Integer keys rather than the natural `PRD-001` strings: smaller in the column
store, faster to join, and they survive a source system renumbering its codes.

**The Unknown member** is the piece that makes a missing key harmless. Every dimension gets a row
with key `-1` labelled `Unknown`, and any fact row whose business key does not resolve points at
it. The relationship stays intact, the revenue stays in the total, and the problem shows up as a
visible category instead of a silent shortfall.

In [ ]:
def add_surrogate_key(df, key_name, order_by):
    """Deterministic integer key, assigned in business-key order."""
    ordering = W.orderBy(*[F.col(c).asc_nulls_last() for c in order_by])
    return df.withColumn(key_name, F.row_number().over(ordering).cast("int"))


def add_unknown_member(df, key_col, overrides=None):
    """Prepend the -1 row. Strings default to 'Unknown', everything else to NULL."""
    overrides = overrides or {}
    values = []
    for field in df.schema.fields:
        if field.name in overrides:
            values.append(overrides[field.name])
        elif field.name == key_col:
            values.append(UNKNOWN_KEY)
        elif isinstance(field.dataType, StringType):
            values.append("Unknown")
        else:
            values.append(None)
    nullable = StructType([StructField(f.name, f.dataType, True) for f in df.schema.fields])
    unknown = spark.createDataFrame([tuple(values)], schema=nullable)
    return unknown.unionByName(df)


def lookup_key(fact, dim, business_col, key_col, dim_business_col=None):
    """Swap a business key for its surrogate, sending anything unresolved to -1."""
    dim_business_col = dim_business_col or business_col
    lookup = (dim.filter(F.col(key_col) != UNKNOWN_KEY)
                 .select(F.col(dim_business_col).alias("_bk"), F.col(key_col).alias("_sk")))
    return (fact.join(F.broadcast(lookup), fact[business_col] == F.col("_bk"), "left")
                .withColumn(key_col, F.coalesce(F.col("_sk"), F.lit(UNKNOWN_KEY)))
                .drop("_bk", "_sk"))


def apply_load_policy(df, label):
    """Duplicates always go. Everything else depends on the policy."""
    before = df.count()
    if "IsDuplicateRow" in df.columns:
        df = df.filter(~F.col("IsDuplicateRow"))
    after_dupes = df.count()
    if GOLD_LOAD_POLICY == "exclude_quarantine" and "DataQualityStatus" in df.columns:
        df = df.filter(F.col("DataQualityStatus") != "Quarantine")
    after = df.count()
    print(f"[{label}] {before:,} silver -> {after:,} gold "
          f"(duplicates removed {before - after_dupes:,}"
          f"{f', quarantined removed {after_dupes - after:,}' if after_dupes != after else ''})")
    return df


def write_gold(df, table_name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table_name))
    count = spark.table(table_name).count()
    print(f"  -> {table_name:30s} {count:>7,} rows")
    if table_name not in GOLD_TABLES:
        GOLD_TABLES.append(table_name)
    return spark.table(table_name)

print("Helpers ready.")

## 3. `gld_dim_date`

Generated rather than derived from the data, so it has no gaps even in months where nothing sold -
a date dimension with holes breaks every time-intelligence measure you will write.

`MonthName` and `MonthYearLabel` each carry a numeric sort column. Without them Power BI sorts
months alphabetically and your chart runs April, August, December.

**Mark this as the date table in the semantic model** (Table tools -> Mark as date table ->
`Date`). `DATESYTD`, `SAMEPERIODLASTYEAR` and friends will not behave until you do.

In [ ]:
dim_date = (spark.sql(
        f"SELECT explode(sequence(to_date('{DATE_FROM}'), to_date('{DATE_TO}'),"
        f" interval 1 day)) AS Date")
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("QuarterNumber", F.quarter("Date"))
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
    .withColumn("YearQuarter", F.concat(F.year("Date"), F.lit(" Q"), F.quarter("Date")))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("MonthNameShort", F.date_format("Date", "MMM"))
    .withColumn("MonthKey", (F.year("Date") * 100 + F.month("Date")).cast("int"))
    .withColumn("MonthYearLabel", F.date_format("Date", "MMM yyyy"))
    .withColumn("MonthStartDate", F.trunc("Date", "MM"))
    .withColumn("MonthEndDate", F.last_day("Date"))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
    .withColumn("DayName", F.date_format("Date", "EEEE"))
    .withColumn("DayOfWeekNumber", F.dayofweek("Date"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin(1, 7))
    .withColumn("IsMonthEnd", F.col("Date") == F.last_day("Date"))
    .select("DateKey", "Date", "Year", "QuarterNumber", "Quarter", "YearQuarter",
            "MonthNumber", "MonthName", "MonthNameShort", "MonthKey", "MonthYearLabel",
            "MonthStartDate", "MonthEndDate", "DayOfMonth", "DayName", "DayOfWeekNumber",
            "IsWeekend", "IsMonthEnd"))

write_gold(dim_date, GOLD_PREFIX + "dim_date")
dim_date.show(3)

## 4. `gld_dim_product`

Silver's data-quality columns do not come to Gold - a business user has no use for
`DataQualityFlags` in a slicer. What does come across is `IsPossibleDuplicate`, because that one
changes how you read a number.

In [ ]:
products_silver = apply_load_policy(
    spark.table(SILVER_PREFIX + "pharma_products"), "products")

dim_product = (products_silver
    .select("ProductID", "ProductName", "BrandName", "TherapeuticArea", "ProductCategory",
            "Manufacturer", "LaunchDate", "LaunchYear", "StandardUnitPriceEUR",
            "StandardUnitCostEUR", "PatentExpiryYear", "IsActive", "IsPatentExpired",
            "ProductLifecycleStage")
    .withColumn("StandardUnitMarginEUR",
                F.col("StandardUnitPriceEUR") - F.col("StandardUnitCostEUR"))
    .withColumn("ProductDisplayName",
                F.concat(F.col("ProductName"), F.lit(" ("), F.col("ProductID"), F.lit(")"))))

dim_product = add_surrogate_key(dim_product, "ProductKey", ["ProductID"])
dim_product = add_unknown_member(dim_product, "ProductKey",
                                 overrides={"ProductID": "UNKNOWN",
                                            "ProductName": "Unknown product",
                                            "ProductDisplayName": "Unknown product"})
dim_product = dim_product.select(
    "ProductKey", "ProductID", "ProductName", "ProductDisplayName", "BrandName",
    "TherapeuticArea", "ProductCategory", "Manufacturer", "LaunchDate", "LaunchYear",
    "StandardUnitPriceEUR", "StandardUnitCostEUR", "StandardUnitMarginEUR",
    "PatentExpiryYear", "IsActive", "IsPatentExpired", "ProductLifecycleStage")

write_gold(dim_product, GOLD_PREFIX + "dim_product")
dim_product.select("ProductKey", "ProductID", "ProductName", "TherapeuticArea",
                   "ProductLifecycleStage").show(10, truncate=False)

## 5. `gld_dim_market`

In [ ]:
markets_silver = apply_load_policy(
    spark.table(SILVER_PREFIX + "pharma_markets"), "markets")

dim_market = (markets_silver
    .select("MarketID", "Country", "Region", "Currency", "MarketManager", "MarketType")
    .withColumn("MarketManager", F.coalesce(F.col("MarketManager"), F.lit("Unassigned")))
    .withColumn("IsEuroMarket", F.col("Currency") == "EUR"))

dim_market = add_surrogate_key(dim_market, "MarketKey", ["MarketID"])
dim_market = add_unknown_member(dim_market, "MarketKey",
                                overrides={"MarketID": "UNKNOWN",
                                           "Country": "Unknown country",
                                           "Region": "Unknown region"})
dim_market = dim_market.select("MarketKey", "MarketID", "Country", "Region", "Currency",
                               "MarketManager", "MarketType", "IsEuroMarket")

write_gold(dim_market, GOLD_PREFIX + "dim_market")
dim_market.show(11, truncate=False)

## 6. `gld_dim_customer`

`Country` and `MarketID` ride along as attributes rather than a foreign key to `gld_dim_market`.
That keeps the model a star instead of a snowflake - Power BI's engine is happier, and users are
not forced to traverse two tables to filter by country.

The two customers Silver quarantined for a missing market are kept. They are real customers with
one missing attribute, and dropping them would orphan their sales.

In [ ]:
customers_silver = apply_load_policy(
    spark.table(SILVER_PREFIX + "pharma_customers"), "customers")

dim_customer = (customers_silver
    .select("CustomerID", "CustomerName", "CustomerType", "CustomerTypeGroup",
            "CustomerSegment", "Country", "MarketID", "IsActive",
            F.col("DuplicateNameCount"))
    .withColumn("MarketID", F.coalesce(F.col("MarketID"), F.lit("UNKNOWN")))
    .withColumn("Country", F.coalesce(F.col("Country"), F.lit("Unknown country")))
    .withColumn("IsPossibleDuplicate", F.col("DuplicateNameCount") > 1)
    .drop("DuplicateNameCount"))

dim_customer = add_surrogate_key(dim_customer, "CustomerKey", ["CustomerID"])
dim_customer = add_unknown_member(dim_customer, "CustomerKey",
                                  overrides={"CustomerID": "UNKNOWN",
                                             "CustomerName": "Unknown customer"})
dim_customer = dim_customer.select(
    "CustomerKey", "CustomerID", "CustomerName", "CustomerType", "CustomerTypeGroup",
    "CustomerSegment", "Country", "MarketID", "IsActive", "IsPossibleDuplicate")

write_gold(dim_customer, GOLD_PREFIX + "dim_customer")
dim_customer.groupBy("CustomerTypeGroup").count().show()

## 7. The small dimensions

`gld_dim_scenario` is the one that makes Actual vs Budget vs Forecast a single measure with a
filter rather than three parallel sets of measures. `ScenarioSortOrder` keeps them in reporting
order instead of alphabetical.

`gld_dim_sales_channel` carries a `Not specified` member because the 2025 file predates the column
- the schema drift is now a visible, labelled category rather than a blank in a slicer.

In [ ]:
dim_scenario = spark.createDataFrame(
    [(1, "Actual", "Reported actuals", 1, True),
     (2, "Budget", "Approved annual budget", 2, False),
     (3, "Forecast", "Latest rolling forecast", 3, False)],
    "ScenarioKey int, ScenarioVersion string, ScenarioDescription string, "
    "ScenarioSortOrder int, IsActual boolean")
dim_scenario = add_unknown_member(dim_scenario, "ScenarioKey",
                                  overrides={"ScenarioVersion": "Unknown",
                                             "ScenarioSortOrder": 99, "IsActual": False})
write_gold(dim_scenario, GOLD_PREFIX + "dim_scenario")

channels = (spark.table(SILVER_PREFIX + "pharma_sales")
            .select("SalesChannel").filter("SalesChannel IS NOT NULL").distinct())
dim_sales_channel = (channels
    .withColumn("ChannelGroup",
                F.when(F.col("SalesChannel").isin("Wholesale", "Tender"), "Indirect")
                 .otherwise("Direct")))
dim_sales_channel = add_surrogate_key(dim_sales_channel, "SalesChannelKey", ["SalesChannel"])
dim_sales_channel = add_unknown_member(
    dim_sales_channel, "SalesChannelKey",
    overrides={"SalesChannel": "Not specified", "ChannelGroup": "Not specified"})
dim_sales_channel = dim_sales_channel.select("SalesChannelKey", "SalesChannel", "ChannelGroup")
write_gold(dim_sales_channel, GOLD_PREFIX + "dim_sales_channel")

# One row per warehouse, not per warehouse-and-market: joining on WarehouseID alone
# against a two-column grain would fan the inventory fact out. Silver's placeholder
# 'UNKNOWN' is excluded so those rows route to the -1 member like any other unresolved key.
dim_warehouse = (spark.table(SILVER_PREFIX + "pharma_inventory")
    .filter("WarehouseID <> 'UNKNOWN'")
    .groupBy("WarehouseID").agg(F.min("MarketID").alias("MarketID"))
    .withColumn("WarehouseName", F.concat(F.lit("Warehouse "), F.col("WarehouseID"))))
dim_warehouse = add_surrogate_key(dim_warehouse, "WarehouseKey", ["WarehouseID"])
dim_warehouse = add_unknown_member(dim_warehouse, "WarehouseKey",
                                   overrides={"WarehouseID": "UNKNOWN",
                                              "WarehouseName": "Unknown warehouse"})
dim_warehouse = dim_warehouse.select("WarehouseKey", "WarehouseID", "WarehouseName", "MarketID")
write_gold(dim_warehouse, GOLD_PREFIX + "dim_warehouse")

dim_expense = (spark.table(SILVER_PREFIX + "pharma_opex")
    .select("ExpenseCategory", "Department").distinct()
    .withColumn("ExpenseGroup",
                F.when(F.col("ExpenseCategory").isin("Sales & Marketing", "Distribution",
                                                     "Medical Affairs"), "Commercial")
                 .when(F.col("ExpenseCategory") == "Research & Development", "R&D")
                 .when(F.col("ExpenseCategory").isin("Manufacturing Overhead"), "Operations")
                 .otherwise("Support"))
    .withColumn("IsEbitdaRelevant", F.lit(True)))
dim_expense = add_surrogate_key(dim_expense, "ExpenseCategoryKey",
                                ["ExpenseCategory", "Department"])
dim_expense = add_unknown_member(dim_expense, "ExpenseCategoryKey",
                                 overrides={"IsEbitdaRelevant": True})
dim_expense = dim_expense.select("ExpenseCategoryKey", "ExpenseCategory", "Department",
                                 "ExpenseGroup", "IsEbitdaRelevant")
write_gold(dim_expense, GOLD_PREFIX + "dim_expense_category")

dim_expense.show(10, truncate=False)

---
## 8. `gld_fact_sales`

Transaction grain, both years unioned. The business keys are swapped for surrogate keys and then
dropped - except `TransactionID`, which stays as a **degenerate dimension** so you can drill to a
single transaction without building a one-row-per-sale dimension table.

`GrossProfitEUR` is **not** created here. It stays a DAX measure, as specified:
`[Net Sales] - [Cost]`. Storing it would bake in an aggregation rule you cannot then change per
visual, and it would be wrong the moment someone filtered to a subset of rows.

In [ ]:
sales_silver = apply_load_policy(spark.table(SILVER_PREFIX + "pharma_sales"), "sales")

fact_sales = sales_silver
fact_sales = lookup_key(fact_sales, spark.table(GOLD_PREFIX + "dim_product"),
                        "ProductID", "ProductKey")
fact_sales = lookup_key(fact_sales, spark.table(GOLD_PREFIX + "dim_market"),
                        "MarketID", "MarketKey")
fact_sales = lookup_key(fact_sales, spark.table(GOLD_PREFIX + "dim_customer"),
                        "CustomerID", "CustomerKey")
fact_sales = lookup_key(fact_sales, spark.table(GOLD_PREFIX + "dim_scenario"),
                        "ScenarioVersion", "ScenarioKey")

# the 2025 file has no SalesChannel - route those rows to the 'Not specified' member
channel_lookup = (spark.table(GOLD_PREFIX + "dim_sales_channel")
                       .select(F.col("SalesChannel").alias("_ch"),
                               F.col("SalesChannelKey").alias("_chKey")))
not_specified = (channel_lookup.filter("_ch = 'Not specified'")
                               .collect()[0]["_chKey"])
fact_sales = (fact_sales
    .join(F.broadcast(channel_lookup), F.col("SalesChannel") == F.col("_ch"), "left")
    .withColumn("SalesChannelKey", F.coalesce(F.col("_chKey"), F.lit(not_specified)))
    .drop("_ch", "_chKey"))

fact_sales = (fact_sales
    .withColumn("DateKey", F.date_format("TransactionDate", "yyyyMMdd").cast("int"))
    .withColumn("HasUnknownKey",
                (F.col("ProductKey") == UNKNOWN_KEY)
                | (F.col("MarketKey") == UNKNOWN_KEY)
                | (F.col("CustomerKey") == UNKNOWN_KEY))
    .withColumn("SourceSystem", F.coalesce(F.col("SourceSystem"), F.lit("EXCEL-2025"))))

fact_sales = fact_sales.select(
    # keys
    "DateKey", "ProductKey", "MarketKey", "CustomerKey", "ScenarioKey", "SalesChannelKey",
    # degenerate dimension
    "TransactionID",
    # measures - local currency
    "UnitsSold", "UnitPriceLocal", "GrossSalesLocal", "DiscountLocal",
    "NetSalesLocal", "CostLocal", "Currency",
    # measures - reporting currency
    "GrossSalesEUR", "DiscountEUR", "NetSalesEUR", "CostEUR",
    # attributes worth keeping on the fact
    "DiscountRatePct", "IsReturn", "IsVolumeOutlier", "HasUnknownKey", "SourceSystem")

write_gold(fact_sales, GOLD_PREFIX + "fact_sales")

print("\nRevenue landing on the Unknown member - visible, not lost:")
(spark.table(GOLD_PREFIX + "fact_sales")
      .groupBy("HasUnknownKey")
      .agg(F.count(F.lit(1)).alias("Rows"),
           F.round(F.sum("NetSalesEUR") / 1e6, 2).alias("NetSalesEURm"))
      .show())

## 9. `gld_fact_plan`

Forecast and budget stacked into one table, told apart by `ScenarioKey`. Two separate fact tables
would mean writing every plan measure twice and a relationship diagram twice as busy.

The measure columns are renamed to a neutral `Plan*` so the union is meaningful -
`RevenueForecastLocal` and `BudgetRevenueLocal` are the same measure under two names.

**Month grain, joined to the date dimension on the first of the month.** So when you put Actual
and Plan in one visual, group by `Year`, `Quarter` or `MonthYearLabel` from `gld_dim_date` - never
by `Date`, or the entire month's plan appears on the 1st beside a single day of actuals.

In [ ]:
forecast_silver = apply_load_policy(
    spark.table(SILVER_PREFIX + "pharma_forecast"), "forecast")
budget_silver = apply_load_policy(
    spark.table(SILVER_PREFIX + "pharma_budget"), "budget")

forecast_part = forecast_silver.select(
    F.col("Month").alias("PlanMonth"), "MonthKey", "ProductID", "MarketID", "ScenarioVersion",
    F.col("UnitsForecast").alias("PlanUnits"),
    F.col("RevenueForecastLocal").alias("PlanRevenueLocal"),
    F.col("CostForecastLocal").alias("PlanCostLocal"),
    "Currency",
    F.col("RevenueForecastEUR").alias("PlanRevenueEUR"),
    F.col("CostForecastEUR").alias("PlanCostEUR"))

budget_part = budget_silver.select(
    F.col("Month").alias("PlanMonth"), "MonthKey", "ProductID", "MarketID", "ScenarioVersion",
    F.col("BudgetUnits").alias("PlanUnits"),
    F.col("BudgetRevenueLocal").alias("PlanRevenueLocal"),
    F.col("BudgetCostLocal").alias("PlanCostLocal"),
    "Currency",
    F.col("BudgetRevenueEUR").alias("PlanRevenueEUR"),
    F.col("BudgetCostEUR").alias("PlanCostEUR"))

fact_plan = forecast_part.unionByName(budget_part)

fact_plan = lookup_key(fact_plan, spark.table(GOLD_PREFIX + "dim_product"),
                       "ProductID", "ProductKey")
fact_plan = lookup_key(fact_plan, spark.table(GOLD_PREFIX + "dim_market"),
                       "MarketID", "MarketKey")
fact_plan = lookup_key(fact_plan, spark.table(GOLD_PREFIX + "dim_scenario"),
                       "ScenarioVersion", "ScenarioKey")

fact_plan = (fact_plan
    .withColumn("DateKey", F.date_format("PlanMonth", "yyyyMMdd").cast("int"))
    .withColumn("PlanUnitPriceEUR",
                F.when(F.col("PlanUnits") > 0,
                       F.round(F.col("PlanRevenueEUR") / F.col("PlanUnits"), 2)))
    .select("DateKey", "MonthKey", "ProductKey", "MarketKey", "ScenarioKey",
            "PlanUnits", "PlanRevenueLocal", "PlanCostLocal", "Currency",
            "PlanRevenueEUR", "PlanCostEUR", "PlanUnitPriceEUR"))

write_gold(fact_plan, GOLD_PREFIX + "fact_plan")

(spark.table(GOLD_PREFIX + "fact_plan").alias("p")
      .join(spark.table(GOLD_PREFIX + "dim_scenario").alias("s"), "ScenarioKey")
      .groupBy("ScenarioVersion")
      .agg(F.count(F.lit(1)).alias("Rows"),
           F.round(F.sum("PlanRevenueEUR") / 1e6, 1).alias("PlanRevenueEURm"))
      .orderBy("ScenarioVersion").show())

## 10. `gld_fact_inventory` and `gld_fact_opex`

`UnitsSold` on the inventory fact is renamed `InventoryUnitsSold`. Two different tables each
carrying a column called `UnitsSold` is how someone ends up summing the wrong one in a measure and
never noticing.

Inventory sits at month **end**, so its `DateKey` is the real snapshot date. OpEx is a month
total, so its `DateKey` is the first of the month, same as the plan.

In [ ]:
inventory_silver = apply_load_policy(
    spark.table(SILVER_PREFIX + "pharma_inventory"), "inventory")

fact_inventory = inventory_silver
fact_inventory = lookup_key(fact_inventory, spark.table(GOLD_PREFIX + "dim_product"),
                            "ProductID", "ProductKey")
fact_inventory = lookup_key(fact_inventory, spark.table(GOLD_PREFIX + "dim_market"),
                            "MarketID", "MarketKey")
fact_inventory = lookup_key(fact_inventory, spark.table(GOLD_PREFIX + "dim_warehouse"),
                            "WarehouseID", "WarehouseKey")

fact_inventory = (fact_inventory
    .withColumn("DateKey", F.date_format("InventoryDate", "yyyyMMdd").cast("int"))
    .select("DateKey", "MonthKey", "ProductKey", "MarketKey", "WarehouseKey",
            "OpeningStock", "UnitsProduced", "UnitsReceived",
            F.col("UnitsSold").alias("InventoryUnitsSold"),
            "UnitsExpired", "ClosingStock", "StockVariance", "StockCoverMonths",
            "InventoryValueLocal", "Currency", "InventoryValueEUR"))

write_gold(fact_inventory, GOLD_PREFIX + "fact_inventory")

opex_silver = apply_load_policy(spark.table(SILVER_PREFIX + "pharma_opex"), "opex")

fact_opex = opex_silver
fact_opex = lookup_key(fact_opex, spark.table(GOLD_PREFIX + "dim_market"),
                       "MarketID", "MarketKey")
fact_opex = lookup_key(fact_opex, spark.table(GOLD_PREFIX + "dim_scenario"),
                       "ScenarioVersion", "ScenarioKey")

expense_lookup = (spark.table(GOLD_PREFIX + "dim_expense_category")
                       .filter(F.col("ExpenseCategoryKey") != UNKNOWN_KEY)
                       .select(F.col("ExpenseCategory").alias("_cat"),
                               F.col("Department").alias("_dept"),
                               F.col("ExpenseCategoryKey").alias("_key")))
fact_opex = (fact_opex
    .join(F.broadcast(expense_lookup),
          (F.col("ExpenseCategory") == F.col("_cat")) & (F.col("Department") == F.col("_dept")),
          "left")
    .withColumn("ExpenseCategoryKey", F.coalesce(F.col("_key"), F.lit(UNKNOWN_KEY)))
    .drop("_cat", "_dept", "_key"))

fact_opex = (fact_opex
    .withColumn("DateKey", F.date_format("Month", "yyyyMMdd").cast("int"))
    .select("DateKey", "MonthKey", "MarketKey", "ExpenseCategoryKey", "ScenarioKey",
            "AmountLocal", "Currency", "AmountEUR", "IsCreditNote"))

write_gold(fact_opex, GOLD_PREFIX + "fact_opex")

## 11. Validate the star

Three things have to be true before this model is safe to build a report on:

1. **No fact key points at a dimension row that does not exist.** In Power BI a broken key does
   not error - it silently creates a blank row in every visual.
2. **Nothing was lost on the way from Silver.** Revenue must tie, to the cent.
3. **The Unknown members are doing their job** rather than quietly swallowing half the data.

In [ ]:
failures = []


def check(name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    if not passed:
        failures.append(name)
    print(f"[{status}] {name}" + (f"   {detail}" if detail else ""))


relationships = [
    ("fact_sales", "DateKey", "dim_date", "DateKey"),
    ("fact_sales", "ProductKey", "dim_product", "ProductKey"),
    ("fact_sales", "MarketKey", "dim_market", "MarketKey"),
    ("fact_sales", "CustomerKey", "dim_customer", "CustomerKey"),
    ("fact_sales", "ScenarioKey", "dim_scenario", "ScenarioKey"),
    ("fact_sales", "SalesChannelKey", "dim_sales_channel", "SalesChannelKey"),
    ("fact_plan", "DateKey", "dim_date", "DateKey"),
    ("fact_plan", "ProductKey", "dim_product", "ProductKey"),
    ("fact_plan", "MarketKey", "dim_market", "MarketKey"),
    ("fact_plan", "ScenarioKey", "dim_scenario", "ScenarioKey"),
    ("fact_inventory", "DateKey", "dim_date", "DateKey"),
    ("fact_inventory", "ProductKey", "dim_product", "ProductKey"),
    ("fact_inventory", "MarketKey", "dim_market", "MarketKey"),
    ("fact_inventory", "WarehouseKey", "dim_warehouse", "WarehouseKey"),
    ("fact_opex", "DateKey", "dim_date", "DateKey"),
    ("fact_opex", "MarketKey", "dim_market", "MarketKey"),
    ("fact_opex", "ExpenseCategoryKey", "dim_expense_category", "ExpenseCategoryKey"),
    ("fact_opex", "ScenarioKey", "dim_scenario", "ScenarioKey"),
]

for fact_name, fact_key, dim_name, dim_key in relationships:
    fact = spark.table(GOLD_PREFIX + fact_name)
    dim = spark.table(GOLD_PREFIX + dim_name).select(F.col(dim_key).alias("_k"))
    broken = fact.join(F.broadcast(dim), fact[fact_key] == F.col("_k"), "left_anti").count()
    nulls = fact.filter(F.col(fact_key).isNull()).count()
    check(f"{fact_name}.{fact_key} -> {dim_name}", broken == 0 and nulls == 0,
          f"{broken} unmatched, {nulls} null")

In [ ]:
# every surrogate key must be unique in its dimension, or the relationship will not build
for dim_name, key_col, business_col in [
        ("dim_date", "DateKey", "Date"),
        ("dim_product", "ProductKey", "ProductID"),
        ("dim_market", "MarketKey", "MarketID"),
        ("dim_customer", "CustomerKey", "CustomerID"),
        ("dim_scenario", "ScenarioKey", "ScenarioVersion"),
        ("dim_sales_channel", "SalesChannelKey", "SalesChannel"),
        ("dim_warehouse", "WarehouseKey", "WarehouseID"),
        ("dim_expense_category", "ExpenseCategoryKey", "ExpenseCategory")]:
    dim = spark.table(GOLD_PREFIX + dim_name)
    total = dim.count()
    distinct_keys = dim.select(key_col).distinct().count()
    check(f"{dim_name}.{key_col} unique", total == distinct_keys,
          f"{total} rows, {distinct_keys} distinct")
    # dim_date is the deliberate exception: Power BI's "Mark as date table" requires a
    # contiguous, non-null date column, and an Unknown member with a NULL Date breaks it.
    # Silver already guarantees every fact row has a parsed date, so none is needed.
    if dim_name != "dim_date":
        has_unknown = dim.filter(F.col(key_col) == UNKNOWN_KEY).count() == 1
        check(f"{dim_name} has exactly one Unknown member", has_unknown)

In [ ]:
# revenue must survive the trip from Silver intact
silver_sales = spark.table(SILVER_PREFIX + "pharma_sales").filter("NOT IsDuplicateRow")
silver_net = float(silver_sales.agg(F.sum("NetSalesEUR")).collect()[0][0])
gold_net = float(spark.table(GOLD_PREFIX + "fact_sales")
                      .agg(F.sum("NetSalesEUR")).collect()[0][0])

check("net sales ties from Silver to Gold", abs(silver_net - gold_net) < 0.01,
      f"silver EUR {silver_net:,.2f} vs gold EUR {gold_net:,.2f}")

# Row counts must tie on every fact. A dimension built at the wrong grain will fan a
# fact out on the join, and nothing else in this notebook would notice.
for fact_name, silver_name in [("fact_sales", "pharma_sales"),
                               ("fact_inventory", "pharma_inventory"),
                               ("fact_opex", "pharma_opex")]:
    expected = spark.table(SILVER_PREFIX + silver_name).filter("NOT IsDuplicateRow").count()
    actual = spark.table(GOLD_PREFIX + fact_name).count()
    check(f"{fact_name} row count ties (no fan-out)", expected == actual,
          f"{expected:,} expected vs {actual:,} loaded")

expected_plan = sum(spark.table(SILVER_PREFIX + n).filter("NOT IsDuplicateRow").count()
                    for n in ["pharma_forecast", "pharma_budget"])
check("fact_plan row count ties (no fan-out)",
      expected_plan == spark.table(GOLD_PREFIX + "fact_plan").count(),
      f"{expected_plan:,} expected")

unknown_net = float(spark.table(GOLD_PREFIX + "fact_sales")
                         .filter("HasUnknownKey")
                         .agg(F.coalesce(F.sum("NetSalesEUR"), F.lit(0))).collect()[0][0])
share = unknown_net / gold_net
print(f"\nRevenue on the Unknown member: EUR {unknown_net:,.0f} ({share:.2%} of net sales)")
check("Unknown member holds a small, reportable share", share < 0.02, f"{share:.2%}")

print("\n" + "=" * 60)
if failures:
    print(f"{len(failures)} CHECK(S) FAILED:")
    for name in failures:
        print("  -", name)
else:
    print("STAR SCHEMA VALID - ready for the semantic model.")
print("=" * 60)

## 12. A first look at the numbers

If these come out sensible, the model works.

In [ ]:
spark.sql(f"""
SELECT  d.Year,
        d.Quarter,
        ROUND(SUM(f.NetSalesEUR) / 1000000, 2) AS NetSalesEURm,
        ROUND(SUM(f.CostEUR)     / 1000000, 2) AS CostEURm,
        ROUND((SUM(f.NetSalesEUR) - SUM(f.CostEUR)) / SUM(f.NetSalesEUR) * 100, 1) AS GrossMarginPct
FROM    {GOLD_PREFIX}fact_sales f
JOIN    {GOLD_PREFIX}dim_date   d ON d.DateKey = f.DateKey
GROUP BY d.Year, d.Quarter
ORDER BY d.Year, d.Quarter
""").show()

spark.sql(f"""
SELECT  p.TherapeuticArea,
        ROUND(SUM(f.NetSalesEUR) / 1000000, 1) AS NetSalesEURm,
        SUM(CASE WHEN f.IsReturn THEN f.UnitsSold ELSE 0 END) AS ReturnedUnits
FROM    {GOLD_PREFIX}fact_sales   f
JOIN    {GOLD_PREFIX}dim_product  p ON p.ProductKey = f.ProductKey
JOIN    {GOLD_PREFIX}dim_date     d ON d.DateKey    = f.DateKey
WHERE   d.Year = 2025
GROUP BY p.TherapeuticArea
ORDER BY NetSalesEURm DESC
""").show(truncate=False)

In [ ]:
# the EBITDA-style bridge the whole model exists to produce
spark.sql(f"""
WITH sales AS (
    SELECT d.Year,
           SUM(f.NetSalesEUR) AS NetSales,
           SUM(f.CostEUR)     AS CostOfSales
    FROM   {GOLD_PREFIX}fact_sales f
    JOIN   {GOLD_PREFIX}dim_date   d ON d.DateKey = f.DateKey
    GROUP BY d.Year
),
opex AS (
    SELECT d.Year, SUM(o.AmountEUR) AS Opex
    FROM   {GOLD_PREFIX}fact_opex     o
    JOIN   {GOLD_PREFIX}dim_date      d ON d.DateKey    = o.DateKey
    JOIN   {GOLD_PREFIX}dim_scenario  s ON s.ScenarioKey = o.ScenarioKey
    WHERE  s.IsActual
    GROUP BY d.Year
)
SELECT  sales.Year,
        ROUND(sales.NetSales / 1000000, 1)                       AS NetSalesEURm,
        ROUND(sales.CostOfSales / 1000000, 1)                    AS CostOfSalesEURm,
        ROUND((sales.NetSales - sales.CostOfSales) / 1000000, 1) AS GrossProfitEURm,
        ROUND(COALESCE(opex.Opex, 0) / 1000000, 1)               AS OpexEURm,
        ROUND((sales.NetSales - sales.CostOfSales
               - COALESCE(opex.Opex, 0)) / 1000000, 1)           AS EbitdaEURm
FROM    sales LEFT JOIN opex ON opex.Year = sales.Year
ORDER BY sales.Year
""").show()

print("Read the 2026 line with care: actual operating expenses exist only for FY2025 "
      "(FY2026 has Budget and Forecast opex, not Actual), and actual sales stop at "
      "31 March. The 2026 figure above is therefore gross profit with no cost base "
      "beneath it - not an EBITDA you can quote.")

---
## 13. Building the semantic model

**Relationships** - all one-to-many from dimension to fact, single direction, dimension filters
fact. Never set any of these to both directions; it creates ambiguity the moment two facts share a
dimension, which here they all do.

| From (one) | To (many) |
|---|---|
| `gld_dim_date[DateKey]` | `gld_fact_sales[DateKey]`, `gld_fact_plan[DateKey]`, `gld_fact_inventory[DateKey]`, `gld_fact_opex[DateKey]` |
| `gld_dim_product[ProductKey]` | `gld_fact_sales`, `gld_fact_plan`, `gld_fact_inventory` |
| `gld_dim_market[MarketKey]` | `gld_fact_sales`, `gld_fact_plan`, `gld_fact_inventory`, `gld_fact_opex` |
| `gld_dim_customer[CustomerKey]` | `gld_fact_sales` |
| `gld_dim_scenario[ScenarioKey]` | `gld_fact_sales`, `gld_fact_plan`, `gld_fact_opex` |
| `gld_dim_sales_channel[SalesChannelKey]` | `gld_fact_sales` |
| `gld_dim_warehouse[WarehouseKey]` | `gld_fact_inventory` |
| `gld_dim_expense_category[ExpenseCategoryKey]` | `gld_fact_opex` |

**Then, in this order:**

1. Mark `gld_dim_date` as the date table on the `Date` column.
2. Sort `MonthName` by `MonthNumber`, `MonthNameShort` by `MonthNumber`, `MonthYearLabel` by
   `MonthKey`, `ScenarioVersion` by `ScenarioSortOrder`. Skip this and every month chart runs in
   alphabetical order.
3. Hide every `*Key` column from report view, and hide `MonthKey` and `DateKey` on the facts.
   Users should never see them; the model needs them.
4. Set `Summarize by` to `Do not summarize` on `PatentExpiryYear` and `LaunchYear` - otherwise
   Power BI cheerfully sums years.
5. Format the EUR columns as currency with zero decimals.

## 14. Starter measures

```dax
Net Sales        = SUM(gld_fact_sales[NetSalesEUR])
Cost of Sales    = SUM(gld_fact_sales[CostEUR])
Gross Profit     = [Net Sales] - [Cost of Sales]
Gross Margin %   = DIVIDE([Gross Profit], [Net Sales])
Units Sold       = SUM(gld_fact_sales[UnitsSold])
Discount %       = DIVIDE(SUM(gld_fact_sales[DiscountEUR]), SUM(gld_fact_sales[GrossSalesEUR]))

Operating Expenses = CALCULATE(SUM(gld_fact_opex[AmountEUR]), gld_dim_scenario[IsActual] = TRUE)
EBITDA             = [Gross Profit] - [Operating Expenses]
EBITDA Margin %    = DIVIDE([EBITDA], [Net Sales])

Plan Revenue     = SUM(gld_fact_plan[PlanRevenueEUR])
Budget Revenue   = CALCULATE([Plan Revenue], gld_dim_scenario[ScenarioVersion] = "Budget")
Forecast Revenue = CALCULATE([Plan Revenue], gld_dim_scenario[ScenarioVersion] = "Forecast")
Variance to Budget   = [Net Sales] - [Budget Revenue]
Variance to Budget % = DIVIDE([Variance to Budget], [Budget Revenue])

Net Sales PY     = CALCULATE([Net Sales], SAMEPERIODLASTYEAR(gld_dim_date[Date]))
Net Sales YoY %  = DIVIDE([Net Sales] - [Net Sales PY], [Net Sales PY])
Net Sales YTD    = TOTALYTD([Net Sales], gld_dim_date[Date])

Closing Stock    = SUM(gld_fact_inventory[ClosingStock])
Returned Units   = CALCULATE([Units Sold], gld_fact_sales[IsReturn] = TRUE)
Return Rate %    = DIVIDE(ABS([Returned Units]), CALCULATE([Units Sold],
                          gld_fact_sales[IsReturn] = FALSE))
Unattributed Sales = CALCULATE([Net Sales], gld_fact_sales[HasUnknownKey] = TRUE)
```

Two notes on those. `Operating Expenses` filters to the actual scenario, otherwise it silently
adds budget and forecast to actual and EBITDA comes out roughly a third too low. And
`Unattributed Sales` is worth a card on the front page - it is the revenue sitting on the Unknown
members, and it should be small and boring. If it moves, the source has changed.

## 15. Two traps specific to this model

**Mixing day-grain and month-grain facts.** `gld_fact_sales` is daily; plan, inventory and opex
are monthly. Group by `Year`, `Quarter` or `MonthYearLabel` when combining them. Group by `Date`
and the whole month's budget lands on the 1st.

**Comparing 2026 actuals to the full-year plan.** Actuals stop at 31 March 2026. Any Actual vs
Budget visual across 2026 will look catastrophic unless you either restrict it to closed months or
write a year-to-date comparison. That is not a data problem, it is the shape of the calendar.